## Example 1: ArbitraryFileIdManager Basic Operations

In [ ]:
import jupyter_server_fileid
from jupyter_server_fileid.manager import ArbitraryFileIdManager
import tempfile
import os

# Example 1: Create and use ArbitraryFileIdManager for file ID operations
print("Example 1: ArbitraryFileIdManager basic operations")

# Create temporary directory for root
with tempfile.TemporaryDirectory() as temp_root:
    # Create manager with temp directory as root
    manager = ArbitraryFileIdManager(db_path=":memory:", root_dir=temp_root)
    
    # Create a test file in the temp directory
    test_file = os.path.join(temp_root, "test_file.txt")
    with open(test_file, 'w') as f:
        f.write("Test file content")
    
    # Index the file to create a file ID
    file_id = manager.index(test_file)
    print(f"File indexed: {os.path.basename(test_file)}")
    print(f"Generated file ID: {file_id}")
    
    # Test lookup operations
    retrieved_id = manager.get_id(test_file)
    retrieved_path = manager.get_path(file_id)
    print(f"ID lookup successful: {retrieved_id == file_id}")
    print(f"Path lookup successful: {retrieved_path == test_file}")
    
    print("File ID operations completed successfully")

## Example 2: File Move and Copy Tracking

In [ ]:
import jupyter_server_fileid
from jupyter_server_fileid.manager import ArbitraryFileIdManager
import tempfile
import os
import shutil

# Example 2: Track file moves and copies with persistent file IDs
print("Example 2: File move and copy tracking")

# Create manager and test files
manager = ArbitraryFileIdManager(db_path=":memory:", root_dir="/tmp")

# Create original file
original_path = tempfile.mktemp(suffix='_original.txt')
with open(original_path, 'w') as f:
    f.write("Original file content")

# Index original file
original_id = manager.index(original_path)
print(f"Original file ID: {original_id}")

# Test file move operation
moved_path = tempfile.mktemp(suffix='_moved.txt')
shutil.move(original_path, moved_path)
move_result = manager.move(original_path, moved_path)
print(f"File moved, ID preserved: {move_result == original_id}")

# Test file copy operation  
copied_path = tempfile.mktemp(suffix='_copied.txt')
shutil.copy2(moved_path, copied_path)
copy_id = manager.copy(moved_path, copied_path)
print(f"File copied, new ID: {copy_id != original_id}")

# Verify mappings
print(f"Moved file path lookup: {manager.get_path(original_id) == moved_path}")
print(f"Copied file has different ID: {copy_id is not None}")

# Clean up
for path in [moved_path, copied_path]:
    if os.path.exists(path):
        os.unlink(path)

print("Move and copy tracking completed")

## Example 3: LocalFileIdManager with Filesystem Metadata

In [ ]:
import jupyter_server_fileid
from jupyter_server_fileid.manager import LocalFileIdManager
import tempfile
import os

# Example 3: Use LocalFileIdManager with filesystem metadata
print("Example 3: LocalFileIdManager with filesystem metadata")

# Create temporary directory for testing
with tempfile.TemporaryDirectory() as temp_dir:
    # Create LocalFileIdManager for the temp directory
    manager = LocalFileIdManager(db_path=":memory:", root_dir=temp_dir)
    
    # Create test file
    test_file = os.path.join(temp_dir, "test_file.txt")
    with open(test_file, 'w') as f:
        f.write("Test content for filesystem tracking")
    
    # Index the file
    file_id = manager.index(test_file)
    print(f"File indexed with ID: {file_id}")
    
    # Get file stats for comparison
    stat_info = os.stat(test_file)
    print(f"File inode: {stat_info.st_ino}")
    print(f"File size: {stat_info.st_size} bytes")
    
    # Test save operation (updates modification time)
    with open(test_file, 'a') as f:
        f.write("\nAdditional content")
    
    save_result = manager.save(test_file)
    print(f"File saved, ID preserved: {save_result == file_id}")
    
    # Verify the file is still tracked
    current_id = manager.get_id(test_file)
    print(f"File still tracked after save: {current_id == file_id}")

print("LocalFileIdManager operations completed")

## Example 4: FileIdExtension Configuration

In [ ]:
import jupyter_server_fileid
from jupyter_server_fileid import FileIdExtension
from jupyter_server_fileid.manager import ArbitraryFileIdManager

# Example 4: Configure FileIdExtension and explore handlers
print("Example 4: FileIdExtension configuration")

# Create extension instance
extension = FileIdExtension()
print(f"Extension class: {type(extension).__name__}")

# Check default configuration safely
default_manager = extension.file_id_manager_class
if hasattr(default_manager, '__name__'):
    print(f"Default manager class: {default_manager.__name__}")
else:
    print(f"Default manager class: {default_manager}")

# Configure with custom manager class
extension.file_id_manager_class = ArbitraryFileIdManager
print(f"Configured manager class: {ArbitraryFileIdManager.__name__}")

# Check available handlers
handlers = extension.handlers
print(f"Number of API handlers: {len(handlers)}")

# Show handler endpoints
for handler_class, pattern in handlers:
    if hasattr(handler_class, '__name__'):
        print(f"  Handler: {handler_class.__name__}")
    else:
        print(f"  Handler: {handler_class}")
    print(f"  Pattern: {pattern}")

# Create manager and check basic functionality
manager = ArbitraryFileIdManager(db_path=":memory:", root_dir="/tmp")
print(f"Manager created: {type(manager).__name__}")

# Test manager methods availability
methods = ['index', 'get_id', 'get_path', 'move', 'copy', 'delete', 'save']
available_methods = [m for m in methods if hasattr(manager, m)]
print(f"Available methods: {', '.join(available_methods)}")

print("Extension configuration completed")

## Example 5: File Deletion and Cleanup Operations

In [ ]:
import jupyter_server_fileid
from jupyter_server_fileid.manager import ArbitraryFileIdManager
import tempfile
import os

# Example 5: File deletion and cleanup operations
print("Example 5: File deletion and cleanup operations")

# Create manager and test files
manager = ArbitraryFileIdManager(db_path=":memory:", root_dir="/tmp")

# Create multiple test files
test_files = []
file_ids = []

for i in range(3):
    temp_file = tempfile.mktemp(suffix=f'_test_{i}.txt')
    with open(temp_file, 'w') as f:
        f.write(f"Test file {i} content")
    
    # Index each file
    file_id = manager.index(temp_file)
    test_files.append(temp_file)
    file_ids.append(file_id)
    print(f"File {i} indexed: {file_id}")

# Verify all files are tracked
print(f"Files tracked before deletion: {len([f for f in file_ids if manager.get_path(f)])}")

# Delete files and clean up tracking
for i, (file_path, file_id) in enumerate(zip(test_files, file_ids)):
    # Remove from file system
    os.unlink(file_path)
    
    # Remove from file ID tracking
    manager.delete(file_path)
    print(f"File {i} deleted and cleaned up")

# Verify cleanup
remaining_files = sum(1 for fid in file_ids if manager.get_path(fid) is not None)
print(f"Files tracked after deletion: {remaining_files}")
print("File deletion and cleanup completed")